In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from pathlib import Path

root = Path('.')
base = pd.read_csv(root/'events_baseline.csv')
current = pd.read_csv(root/'events_raw.csv')
print(len(base), len(current))
base.head(2)

## PSI helper and drift computation
PSI is a simple binned divergence score. Rough rule of thumb: >0.1 small shift, >0.25 moderate, >0.5 significant.

In [ ]:
def psi(expected, actual, bins=10):
    e_hist, edges = np.histogram(expected, bins=bins)
    a_hist, _ = np.histogram(actual, bins=edges)
    e_ratio = np.clip(e_hist / max(1,len(expected)), 1e-6, None)
    a_ratio = np.clip(a_hist / max(1,len(actual)), 1e-6, None)
    return float(np.sum((a_ratio - e_ratio) * np.log(a_ratio / e_ratio)))

numeric = ['bytes_transferred','ip_reputation_score','session_length_sec']
psi_scores = {c: psi(base[c].values, current[c].values) for c in numeric}
ks_pvals = {c: float(stats.ks_2samp(base[c].values, current[c].values).pvalue) for c in numeric}
psi_scores, ks_pvals

## Flag drift and save a small report
We'll flag any feature with PSI > 0.25 or KS p-value < 0.01 as drifted.

In [ ]:
import json
drifted = []
for c in numeric:
    if psi_scores[c] > 0.25 or ks_pvals[c] < 0.01:
        drifted.append(c)
report = { 'psi': psi_scores, 'ks_pvalue': ks_pvals, 'drifted_features': drifted }
with open(root/'drift_report.json','w') as f: json.dump(report,f,indent=2)
report